# SOXX 2019 old-form to N-PORT transition index

固定计划：`plan.json`。仅核对 2019 年两份旧表单 filing index 的报告期、申报时钟、系列身份和原件入口；市场收益读取、拟合及账户均为 0。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd(); card=root/'research/stage1_soxx_2019_transition_index_20261023'; plan=json.loads((card/'plan.json').read_text())
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
for name,key in [('series_NQ','series_NQ_sha256'),('shareholder_forms','shareholder_forms_sha256')]: assert sha(root/plan['parent_sources'][name])==plan['parent_sources'][key]
raw=root/'data/snapshots/stage1_soxx_2019_transition_index_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];error=None
for i,spec in enumerate(plan['exact_index_pages']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 accession=spec['accession'];body=raw/(accession+'.html');headers=raw/(accession+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['request']['connect_timeout_seconds']),'--max-time',str(plan['request']['total_timeout_seconds']),'--max-filesize',str(plan['request']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'form':spec['form'],'accession':accession,'expected_filed':spec['filed'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'form':spec['form'],'accession':accession,'expected_filed':spec['filed'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>plan['request']['max_response_bytes_each']:error='transport_or_size_failure';break
 if i+1<len(plan['exact_index_pages']):time.sleep(plan['request']['minimum_seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-2019-transition-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-2019-transition-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
transport={'status':'two_old_form_2019_indexes_archived_parse_pending' if not error and len(receipts)==2 else 'partial_index_transport_needs_repair','error':error,'GET_attempts':state['GET_attempts'],'HTTP200_pages':sum(x.get('http_status')==200 and x.get('curl_exit_code')==0 for x in receipts),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'transport_result.json').write_text(json.dumps(transport,indent=2)+'\n')
if transport['status']!='two_old_form_2019_indexes_archived_parse_pending': print(transport);raise SystemExit(0)
rows=[];errors=[]
for spec in plan['exact_index_pages']:
 receipt=next(x for x in receipts if x['accession']==spec['accession']);path=raw/(spec['accession']+'.html');assert sha(path)==receipt['raw_sha256'];soup=BeautifulSoup(path.read_bytes(),'html.parser')
 try:
  secnum=soup.select_one('#secNum');assert secnum and spec['accession'] in secnum.get_text(' ',strip=True)
  formname=soup.select_one('#formName');assert formname and spec['form'] in formname.get_text(' ',strip=True)
  info={}
  for group in soup.select('div.formContent div.formGrouping'):
   children=group.find_all('div',recursive=False)
   for j,node in enumerate(children[:-1]):
    if 'infoHead' in (node.get('class') or []):info[node.get_text(' ',strip=True)]=children[j+1].get_text(' ',strip=True)
  filed=info.get('Filing Date');accepted=info.get('Accepted');period=info.get('Period of Report')
  assert filed==spec['filed'] and re.fullmatch(r'20\d{2}-\d{2}-\d{2}',filed)
  assert accepted and re.fullmatch(r'20\d{2}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}',accepted)
  assert period and re.fullmatch(r'20\d{2}-\d{2}-\d{2}',period) and period<=filed
  div=soup.select_one('#seriesDiv');series_text=div.get_text(' ',strip=True) if div else ''
  docs=[]
  for tr in soup.select('table.tableFile tr'):
   cells=tr.find_all('td',recursive=False)
   if len(cells)<5:continue
   link=cells[2].find('a')
   if link:docs.append({'document':link.get_text(' ',strip=True),'href':link.get('href'),'type':cells[3].get_text(' ',strip=True),'size_text':cells[4].get_text(' ',strip=True)})
  primary=[x for x in docs if x['type']==spec['form'] and x['document'].lower().endswith(('.htm','.html'))]
  assert len(primary)==1
  rows.append({'accession':spec['accession'],'form':spec['form'],'filed':filed,'accepted_display':accepted,'accepted_display_timezone':'SEC_page_unverified','report_period':period,'series_text':series_text,'series_id_present':'S000004354' in series_text,'class_id_present':'C000012084' in series_text,'ticker_present':bool(re.search(r'\bSOXX\b',series_text)),'primary_document':primary[0],'source_index_sha256':receipt['raw_sha256']})
 except Exception as exc:errors.append({'accession':spec['accession'],'error':type(exc).__name__+':'+str(exc)[:150]})
result={'status':'two_2019_transition_index_metadata_qualified_bodies_pending' if not errors and len(rows)==2 else 'index_metadata_mismatch_stop','source_plan_sha256':sha(card/'plan.json'),'source_archive_sha256':digest,'source_NAS_scoped_restore_verified':True,'expected_pages':2,'parsed_pages':len(rows),'errors':errors,'rows':rows,'series_identity_qualified_from_index_only':len(rows)==2 and all(x['series_id_present'] and x['class_id_present'] and x['ticker_present'] for x in rows),'accepted_clock_timezone_qualified':False,'original_holdings_values_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'metadata_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({'transport':transport['status'],'metadata':result['status'],'rows':rows,'errors':errors})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'transport': 'two_old_form_2019_indexes_archived_parse_pending', 'metadata': 'two_2019_transition_index_metadata_qualified_bodies_pending', 'rows': [{'accession': '0001193125-19-059323', 'form': 'N-Q', 'filed': '2019-03-01', 'accepted_display': '2019-03-01 08:18:09', 'accepted_display_timezone': 'SEC_page_unverified', 'report_period': '2018-12-31', 'series_text': 'Series and Classes/Contracts Information: Existing Status Name Ticker Symbol CIK 0001100663 Series S000004305 iShares Global Tech ETF Class/Contract C000012035 iShares Global Tech ETF IXN Status Name Ticker Symbol CIK 0001100663 Series S000004306 iShares S&P 100 ETF Class/Contract C000012036 iShares S&P 100 ETF OEF Status Name Ticker Symbol CIK 0001100663 Series S000004307 iShares Core S&P Mid-Cap ETF Class/Contract C000012037 iShares Core S&P Mid-Cap ETF IJH Status Name Ticker Symbol CIK 0001100663 Series S000004308 iShares S&P Mid-Cap 400 Growth ETF Class/Contract C000012038 iShares S&P Mid-Cap 400 Growth ETF IJK Status Na

## 中文结论

两份 SEC 旧表单索引 GET、NAS 远端哈希和限定恢复通过；报告期分别为 2018-12-31、2019-03-31。第一次多系列表扫描仅证明字段各自存在；冻结的零网络修复 `identity_repair.ipynb` 进一步确认 S000004354 系列与 C000012084/SOXX 类别在两页逐行相邻。正文未读取，持仓值和完整 Stage1 特征仍未知；0 收益读取、0 拟合、0 账户。决定：`two_2019_old_form_SOXX_index_identities_qualified_bodies_pending`。下一步按原件大小另行冻结正文来源资格。